In [1]:
%%writefile device_query.cu
#include <iostream>
#include <cuda_runtime.h>

int main() {
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);

    for (int i = 0; i < deviceCount; i++) {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);
        printf("Device Number: %d\n", i);
        printf("  Device name: %s\n", prop.name);
        printf("  Compute capability: %d.%d\n", prop.major, prop.minor);
        printf("  Max Grid Dimensions: [%d, %d, %d]\n", prop.maxGridSize[0], prop.maxGridSize[1], prop.maxGridSize[2]);
        printf("  Max Block Dimensions: [%d, %d, %d]\n", prop.maxThreadsDim[0], prop.maxThreadsDim[1], prop.maxThreadsDim[2]);
        printf("  Max threads per block: %d\n", prop.maxThreadsPerBlock);
        printf("  Warp size: %d\n", prop.warpSize);
        printf("  Total Global Memory: %zu bytes\n", prop.totalGlobalMem);
        printf("  Shared Memory per block: %zu bytes\n", prop.sharedMemPerBlock);
        printf("  Constant Memory: %zu bytes\n", prop.totalConstMem);
        printf("  Double precision support: %s\n", (prop.major >= 1 ? "Yes" : "No"));
    }
    return 0;
}

Writing device_query.cu


In [2]:
!nvcc device_query.cu -o device_query
!./device_query

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Device Number: 0
  Device name: Tesla T4
  Compute capability: 7.5
  Max Grid Dimensions: [2147483647, 65535, 65535]
  Max Block Dimensions: [1024, 1024, 64]
  Max threads per block: 1024
  Warp size: 32
  Total Global Memory: 15637086208 bytes
  Shared Memory per block: 49152 bytes
  Constant Memory: 65536 bytes
  Double precision support: Yes


In [3]:
%%writefile array_sum.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void sumKernel(float* d_in, float* d_out, int n) {
    int tid = threadIdx.x + blockIdx.x * blockDim.x;
    if (tid < n) {
        atomicAdd(d_out, d_in[tid]);
    }
}

int main() {
    int n = 1 << 20; // 1 million elements
    size_t size = n * sizeof(float);

    float *h_in = (float*)malloc(size);
    float h_out = 0.0f;
    for(int i=0; i<n; i++) h_in[i] = 1.0f;

    float *d_in, *d_out;
    cudaMalloc(&d_in, size);
    cudaMalloc(&d_out, sizeof(float));

    cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_out, &h_out, sizeof(float), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;

    sumKernel<<<blocksPerGrid, threadsPerBlock>>>(d_in, d_out, n);

    cudaMemcpy(&h_out, d_out, sizeof(float), cudaMemcpyDeviceToHost);

    printf("Array Sum: %f\n", h_out);

    cudaFree(d_in);
    cudaFree(d_out);
    free(h_in);
    return 0;
}

Writing array_sum.cu


In [4]:
!nvcc array_sum.cu -o array_sum && ./array_sum

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Array Sum: 1048576.000000


In [1]:
%%writefile matrix_add.cu
#include <iostream>
#include <cuda_runtime.h>

// CUDA Kernel for Matrix Addition
__global__ void matrixAdd(int* A, int* B, int* C, int N) {
    // Calculate global row and column index
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Boundary check to ensure we don't access memory outside the matrix
    if (row < N && col < N) {
        int idx = row * N + col; // 2D index to 1D index mapping
        C[idx] = A[idx] + B[idx];
    }
}

int main() {
    int N = 1024; // Matrix size N x N
    size_t size = N * N * sizeof(int);

    // 1. Allocate Host memory
    int *h_A = (int*)malloc(size);
    int *h_B = (int*)malloc(size);
    int *h_C = (int*)malloc(size);

    // Initialize matrices
    for(int i=0; i<N*N; i++) {
        h_A[i] = 10; // Example value
        h_B[i] = 20; // Example value
    }

    // 2. Allocate Device memory
    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    // 3. Copy host memory to device
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // 4. Initialize thread block and kernel grid dimensions
    dim3 threadsPerBlock(16, 16); // 16x16 = 256 threads per block
    dim3 blocksPerGrid((N + 15) / 16, (N + 15) / 16);

    // 5. Invoke CUDA kernel
    matrixAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    // 6. Copy results from device to host
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // Verify a result
    printf("Result at C[0]: %d (Expected: 30)\n", h_C[0]);
    printf("Matrix Addition completed successfully!\n");

    // 7. Free device memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Overwriting matrix_add.cu


In [2]:
!nvcc matrix_add.cu -o matrix_add
!./matrix_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Result at C[0]: 30 (Expected: 30)
Matrix Addition completed successfully!
